# LensWord — Text Preprocessing

This notebook takes the cleaned Amazon reviews dataset and prepares it for the
LSTM model. It handles tokenization, vocabulary building, converting words to
numbers, padding sequences to equal length, and splitting the data into
train/validation/test sets. The output is saved as PyTorch tensors ready for
model training.

Before running: make sure `01_EDA_lensword.ipynb` has been run first and that
`amazon_reviews_cleaned.csv` exists inside the `data/` folder.

In [1]:
# Import all necessary libraries
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
import sys
import os

# Add project root to path so we can import config.py
sys.path.append(os.path.abspath('..'))
from src.config import *

In [2]:
# Load the cleaned dataset from EDA notebook
df = pd.read_csv('../data/amazon_reviews_cleaned.csv')

# Confirm it loaded correctly
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
df.head()

Dataset Shape: (15149, 2)

First 5 rows:


,verified_reviews,sentiment
0,They used to be delicious. I live right around...,Negative
1,I like everything about it !!!,Positive
2,You walk into the restaurant and the buffet lo...,Neutral
3,I really liked this place. The food was great...,Positive
4,I use the product as an alarm clock. I love be...,Positive


In [3]:
# Load additional data from HuggingFace to expand our training set
from datasets import load_dataset

print("Loading dataset from HuggingFace...")
dataset = load_dataset("Yelp/yelp_review_full", split="train")
print(f"Dataset loaded! Total samples: {len(dataset)}")

# Convert to pandas dataframe
df_hf = dataset.to_pandas()
print(f"\nColumns: {df_hf.columns.tolist()}")
print(f"\nSample:")
print(df_hf.head())


Loading dataset from HuggingFace...


Dataset loaded! Total samples: 650000

Columns: ['label', 'text']

Sample:
   label                                               text
0      4  dr. goldberg offers everything i look for in a...
1      1  Unfortunately, the frustration of being Dr. Go...
2      3  Been going to Dr. Goldberg for over 10 years. ...
3      3  Got a letter in the mail last week that said D...
4      0  I don't know what Dr. Goldberg was like before...


In [4]:
# Check label distribution
print("Yelp Label Distribution:")
print(df_hf['label'].value_counts().sort_index())
print("\nLabel meaning: 0=1star, 1=2star, 2=3star, 3=4star, 4=5star")

# Map to our 3-class sentiment system
# 0-1 (1-2 stars) -> Negative
# 2 (3 stars) -> Neutral  
# 3-4 (4-5 stars) -> Positive
def map_yelp_sentiment(label):
    if label <= 1:
        return 'Negative'
    elif label == 2:
        return 'Neutral'
    else:
        return 'Positive'

df_hf['sentiment'] = df_hf['label'].apply(map_yelp_sentiment)
df_hf = df_hf.rename(columns={'text': 'verified_reviews'})

print("\nAfter mapping:")
print(df_hf['sentiment'].value_counts())


Yelp Label Distribution:
label
0    130000
1    130000
2    130000
3    130000
4    130000
Name: count, dtype: int64

Label meaning: 0=1star, 1=2star, 2=3star, 3=4star, 4=5star

After mapping:
sentiment
Positive    260000
Negative    260000
Neutral     130000
Name: count, dtype: int64


In [5]:
# Sample a balanced subset from Yelp dataset
# Filter to short reviews only to match Amazon Alexa review style
import pandas as pd

# Keep only reviews similar in length to our Amazon data (max 50 words)
df_hf['word_count'] = df_hf['verified_reviews'].apply(lambda x: len(str(x).split()))
df_hf_short = df_hf[df_hf['word_count'] <= 50]

print(f"Short reviews available: {len(df_hf_short)}")
print(f"Distribution:")
print(df_hf_short['sentiment'].value_counts())

sample_size = 2000

df_yelp_sampled = pd.concat([
    df_hf_short[df_hf_short['sentiment'] == 'Negative'].sample(sample_size, random_state=42),
    df_hf_short[df_hf_short['sentiment'] == 'Neutral'].sample(sample_size, random_state=42),
    df_hf_short[df_hf_short['sentiment'] == 'Positive'].sample(sample_size, random_state=42)
])

df_yelp_sampled = df_yelp_sampled[['verified_reviews', 'sentiment']].reset_index(drop=True)

print(f"\nSampled Yelp dataset (short reviews only):")
print(df_yelp_sampled['sentiment'].value_counts())
print(f"\nTotal sampled: {len(df_yelp_sampled)}")
print(f"\nSample review:")
print(df_yelp_sampled['verified_reviews'][0])

Short reviews available: 154809
Distribution:
sentiment
Positive    77095
Negative    51198
Neutral     26516
Name: count, dtype: int64

Sampled Yelp dataset (short reviews only):
sentiment
Negative    2000
Neutral     2000
Positive    2000
Name: count, dtype: int64

Total sampled: 6000

Sample review:
Came here a few times and enjoyed it. However, I've been coming by the last few Saturday evenings and have never found it open, despite the window claiming openness on Saturdays. The lack of predictable hours (and lack of ANY posted hours) makes this a poor choice for gelato.


In [6]:
# Combine Amazon Alexa reviews with Yelp reviews
# Load our original cleaned Amazon data first
df_amazon = pd.read_csv('../data/amazon_reviews_cleaned.csv')

print("Before combining:")
print(f"Amazon Alexa reviews: {len(df_amazon)}")
print(f"Yelp reviews: {len(df_yelp_sampled)}")

# Combine both datasets
df_combined = pd.concat([df_amazon, df_yelp_sampled], ignore_index=True)

# Shuffle the combined dataset
df_combined = df_combined.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\nAfter combining:")
print(f"Total reviews: {len(df_combined)}")
print(f"\nSentiment distribution:")
print(df_combined['sentiment'].value_counts())

Before combining:
Amazon Alexa reviews: 15149
Yelp reviews: 6000

After combining:
Total reviews: 21149

Sentiment distribution:
sentiment
Positive    8741
Negative    6256
Neutral     6152
Name: count, dtype: int64


In [7]:
# Save the combined dataset
df_combined.to_csv('../data/amazon_reviews_cleaned.csv', index=False)
print("Combined dataset saved to data/amazon_reviews_cleaned.csv")
print(f"Total reviews: {len(df_combined)}")
print(f"\nFinal distribution:")
print(df_combined['sentiment'].value_counts())
print(f"\nPercentages:")
print((df_combined['sentiment'].value_counts(normalize=True) * 100).round(2))

Combined dataset saved to data/amazon_reviews_cleaned.csv
Total reviews: 21149

Final distribution:
sentiment
Positive    8741
Negative    6256
Neutral     6152
Name: count, dtype: int64

Percentages:
sentiment
Positive    41.33
Negative    29.58
Neutral     29.09
Name: proportion, dtype: float64


In [8]:
# Convert sentiment labels to numbers
# The model needs numbers not words
label_map = {'Positive': 2, 'Neutral': 1, 'Negative': 0}
df['label'] = df['sentiment'].map(label_map)

# Confirm the mapping
print("Label Mapping:", label_map)
print("\nSample:")
df[['sentiment', 'label']].head(10)

Label Mapping: {'Positive': 2, 'Neutral': 1, 'Negative': 0}

Sample:


,sentiment,label
0,Negative,0
1,Positive,2
2,Neutral,1
3,Positive,2
4,Positive,2
5,Negative,0
6,Neutral,1
7,Neutral,1
8,Neutral,1
9,Negative,0


In [9]:
# Clean the review text
# Remove punctuation, numbers, and extra spaces
# Convert everything to lowercase

def clean_text(text):
    text = str(text).lower()                    # lowercase
    text = re.sub(r'[^a-z\s]', '', text)        # remove punctuation and numbers
    text = re.sub(r'\s+', ' ', text).strip()    # remove extra spaces
    return text

df['cleaned_review'] = df['verified_reviews'].apply(clean_text)

# Show before and after
print("Before:", df['verified_reviews'][2])
print("\nAfter: ", df['cleaned_review'][2])

Before: You walk into the restaurant and the buffet looks really delicious with an overwhelming variety. The highlights were the mango ice cream, grilled fish, vegetable pakora and the homemade chutneys. The service was very friendly and attentive. The food is all way too mild. There should be, at least, one dish in an Indian restaurant that threatens to burn your face off. The dishes all look good but they left me very unsatisfied. In addition, all of the dishes seemed to be off temperature. It wasn't bad, but I am not ovewhelmed. The chicken tika masala was tough, the sauce was not thick enough and was, as I mentioned earlier, way too mild. I have a hunch, had I ordered off of the menu, the food would have been much better. The bread was more like pita and less like Naan and that's a shame because the bread is very important to me, for an Indian meal. There was a cabbage dish that verged on being fantastic. I probably will not have a Sunday afternoon craving for this place.  In the e

In [10]:
# Build a vocabulary from all the words in the dataset
# We assign a unique number to each word

# Tokenize all reviews into words
all_words = []
for review in df['cleaned_review']:
    all_words.extend(review.split())

# Count word frequencies
word_counts = Counter(all_words)
print(f"Total unique words found: {len(word_counts)}")

# Keep only the most common words
vocab = ['<PAD>', '<UNK>'] + [word for word, count in word_counts.most_common(MAX_VOCAB_SIZE - 2)]
word2idx = {word: idx for idx, word in enumerate(vocab)}

print(f"Vocabulary size: {len(vocab)}")
print(f"\nSample words from vocabulary:")
print(list(word2idx.items())[:10])

Total unique words found: 34433
Vocabulary size: 4340

Sample words from vocabulary:
[('<PAD>', 0), ('<UNK>', 1), ('the', 2), ('and', 3), ('i', 4), ('a', 5), ('to', 6), ('was', 7), ('of', 8), ('it', 9)]


In [11]:
# Convert each review from words to a list of numbers
# using the vocabulary we just built

def text_to_sequence(text):
    words = text.split()
    sequence = [word2idx.get(word, word2idx['<UNK>']) for word in words]
    return sequence

df['sequence'] = df['cleaned_review'].apply(text_to_sequence)

# Show before and after
print("Review:  ", df['cleaned_review'][0])
print("\nSequence:", df['sequence'][0])

Review:   they used to be delicious i live right around the corner and our family would eat there all the time when we wanted a quick dinner and didnt want to have to make it at home the last two times i have been there though it has been horrible the service is horrible the people are rude and i actually got sick the last two times i ate there the first time i got sick i didnt realize it was the food i thought i just had a stomach bug and it wasnt nearly as bad as the second time i got sick the very next time after that i got very sick again it was one of the worst food poisoning ive had since i was a kid there also seems to be a foul smell inside a mixture between them not cleaning out their grease traps properly raw sewage and a dirty restaurant with mildew floors as far as the rude service goes my friend and i were ordering take out a while back and my buddy thought he had paid the whole balance but apparently he was off by dollars which was purely accident the lady in the front st

In [12]:
# Pad all sequences to the same length (MAX_SEQ_LENGTH = 50)
# Short reviews get zeros added at the end
# Long reviews get cut off at 50 words

def pad_sequence(sequence, max_length):
    if len(sequence) < max_length:
        # Add zeros at the end to reach max_length
        sequence = sequence + [0] * (max_length - len(sequence))
    else:
        # Cut off at max_length
        sequence = sequence[:max_length]
    return sequence

df['padded_sequence'] = df['sequence'].apply(lambda x: pad_sequence(x, MAX_SEQ_LENGTH))

# Confirm all sequences are the same length
lengths = df['padded_sequence'].apply(len)
print(f"All sequences same length: {lengths.nunique() == 1}")
print(f"Sequence length: {lengths[0]}")
print(f"\nSample padded sequence:")
print(df['padded_sequence'][0])

All sequences same length: True
Sequence length: 50

Sample padded sequence:
[21, 255, 6, 30, 204, 4, 508, 146, 144, 2, 1049, 3, 46, 419, 51, 151, 35, 39, 2, 53, 50, 18, 206, 5, 437, 211, 3, 87, 129, 6, 23, 6, 136, 9, 25, 205, 2, 194, 123, 166, 4, 23, 70, 35, 157, 9, 92, 70, 509, 2]


In [13]:
# Split data into train, validation and test sets
# 75% training, 15% validation, 10% testing
from sklearn.model_selection import train_test_split

X = np.array(df['padded_sequence'].tolist())
y = np.array(df['label'].tolist())

# First split off 25% for val+test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

# Split the 25% into 15% val and 10% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.4, random_state=42, stratify=y_temp)

print(f"Training set:   {X_train.shape[0]} reviews")
print(f"Validation set: {X_val.shape[0]} reviews")
print(f"Test set:       {X_test.shape[0]} reviews")


Training set:   11361 reviews
Validation set: 2272 reviews
Test set:       1516 reviews


In [14]:
# Convert numpy arrays to PyTorch tensors
# This is the format PyTorch needs for training

X_train_tensor = torch.tensor(X_train, dtype=torch.long)
X_val_tensor = torch.tensor(X_val, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor = torch.tensor(y_val, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print("Tensors created successfully!")
print(f"X_train shape: {X_train_tensor.shape}")
print(f"X_val shape:   {X_val_tensor.shape}")
print(f"X_test shape:  {X_test_tensor.shape}")

Tensors created successfully!
X_train shape: torch.Size([11361, 50])
X_val shape:   torch.Size([2272, 50])
X_test shape:  torch.Size([1516, 50])


In [15]:
# Create PyTorch Datasets and DataLoaders
# DataLoader feeds data to the model in batches during training

from torch.utils.data import TensorDataset, DataLoader

# Wrap tensors into datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset = TensorDataset(X_val_tensor, y_val_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders created successfully!")
print(f"Training batches:   {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches:       {len(test_loader)}")

DataLoaders created successfully!
Training batches:   356
Validation batches: 71
Test batches:       48


In [16]:
# Save all preprocessed data so we can load it in the training notebook
import pickle

# Save tensors
torch.save(X_train_tensor, '../data/X_train.pt')
torch.save(X_val_tensor, '../data/X_val.pt')
torch.save(X_test_tensor, '../data/X_test.pt')
torch.save(y_train_tensor, '../data/y_train.pt')
torch.save(y_val_tensor, '../data/y_val.pt')
torch.save(y_test_tensor, '../data/y_test.pt')

# Save vocabulary
with open('../data/word2idx.pkl', 'wb') as f:
    pickle.dump(word2idx, f)

print("All preprocessed data saved successfully!")
print("Files saved to data/ folder:")
print("  - X_train.pt, X_val.pt, X_test.pt")
print("  - y_train.pt, y_val.pt, y_test.pt")
print("  - word2idx.pkl")

All preprocessed data saved successfully!
Files saved to data/ folder:
  - X_train.pt, X_val.pt, X_test.pt
  - y_train.pt, y_val.pt, y_test.pt
  - word2idx.pkl


In [17]:
# Final summary of preprocessing
print("=" * 40)
print("PREPROCESSING SUMMARY - LensWord")
print("=" * 40)
print(f"Total Reviews:        {len(df)}")
print(f"Vocabulary Size:      {len(vocab)}")
print(f"Max Sequence Length:  {MAX_SEQ_LENGTH}")
print(f"Training Set:         {X_train_tensor.shape[0]}")
print(f"Validation Set:       {X_val_tensor.shape[0]}")
print(f"Test Set:             {X_test_tensor.shape[0]}")
print(f"Batch Size:           {BATCH_SIZE}")
print(f"Training Batches:     {len(train_loader)}")
print("=" * 40)

PREPROCESSING SUMMARY - LensWord
Total Reviews:        15149
Vocabulary Size:      4340
Max Sequence Length:  50
Training Set:         11361
Validation Set:       2272
Test Set:             1516
Batch Size:           32
Training Batches:     356


In [18]:
import importlib
import imblearn
print("imblearn version:", imblearn.__version__)

imblearn version: 0.14.2


In [19]:
import sys
print(sys.executable)

C:\Users\Administrator\Desktop\Apeiron_ML_2026\Project 07\lensword\.venv\Scripts\python.exe


In [20]:
# Apply SMOTE to fix remaining class imbalance in training set only
# We never apply SMOTE to validation or test sets
from imblearn.over_sampling import SMOTE

print("Before SMOTE:")
unique, counts = np.unique(y_train, return_counts=True)
labels = ['Negative', 'Neutral', 'Positive']
for u, c in zip(unique, counts):
    print(f"  {labels[u]}: {c}")

# Apply full SMOTE to balance all classes
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
unique, counts = np.unique(y_train_smote, return_counts=True)
for u, c in zip(unique, counts):
    print(f"  {labels[u]}: {c}")

print(f"\nTraining set size before SMOTE: {len(X_train)}")
print(f"Training set size after SMOTE:  {len(X_train_smote)}")

Before SMOTE:
  Negative: 3192
  Neutral: 3114
  Positive: 5055

After SMOTE:
  Negative: 5055
  Neutral: 5055
  Positive: 5055

Training set size before SMOTE: 11361
Training set size after SMOTE:  15165


In [21]:
# Convert SMOTE output to PyTorch tensors and save
X_train_smote_tensor = torch.tensor(X_train_smote, dtype=torch.long)
y_train_smote_tensor = torch.tensor(y_train_smote, dtype=torch.long)

# Save new balanced training tensors
torch.save(X_train_smote_tensor, '../data/X_train_smote.pt')
torch.save(y_train_smote_tensor, '../data/y_train_smote.pt')

print("Balanced training tensors saved!")
print(f"X_train_smote shape: {X_train_smote_tensor.shape}")
print(f"y_train_smote shape: {y_train_smote_tensor.shape}")

Balanced training tensors saved!
X_train_smote shape: torch.Size([15165, 50])
y_train_smote shape: torch.Size([15165])


In [22]:
import torch
X_check = torch.load('../data/X_train_smote.pt')
y_check = torch.load('../data/y_train_smote.pt')

import numpy as np
unique, counts = np.unique(y_check.numpy(), return_counts=True)
labels = ['Negative', 'Neutral', 'Positive']
print("Currently saved on disk:")
for u, c in zip(unique, counts):
    print(f"  {labels[u]}: {c}")

Currently saved on disk:
  Negative: 5055
  Neutral: 5055
  Positive: 5055
